In [ ]:
pip install librosa matplotlib

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
audio_path = Path("file_wav/ID10.StudioRuggi.wav")

audio, sr = sf.read(audio_path)

print(audio.shape)
print(sr)

#stereo, perché ha forma (campioni, 2);
#composto da 9.751.552 campioni per canale;
#campionato a 48.000 Hz;
#lungo circa 203,16 secondi, cioè poco più di 3 minuti.

In [ ]:
left = audio[:, 0]
right = audio[:, 1]

print("Canale sinistro:", left.shape)
print("Canale destro:", right.shape)

In [ ]:
time = np.arange(len(left)) / sr

plt.figure(figsize=(16, 4))
plt.plot(time, left)
plt.xlabel("Tempo [s]")
plt.ylabel("Ampiezza")
plt.title("ID10 - Canale sinistro")
plt.savefig("img/ID10_canale_sinistro.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(16, 4))
plt.plot(time, right)
plt.xlabel("Tempo [s]")
plt.ylabel("Ampiezza")
plt.title("ID10 - Canale destro")
plt.savefig("img/ID10_canale_destro.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from IPython.display import Audio, display

durata_ascolto = 60
n_campioni = durata_ascolto * sr

print("Canale sinistro")
display(Audio(left[:n_campioni], rate=sr))

print("Canale destro")
display(Audio(right[:n_campioni], rate=sr))

In [ ]:
correlazione = np.corrcoef(left, right)[0, 1]

rms_left = np.sqrt(np.mean(left ** 2))
rms_right = np.sqrt(np.mean(right ** 2))

print(f"Correlazione tra i canali: {correlazione:.4f}")
print(f"RMS canale sinistro: {rms_left:.6f}")
print(f"RMS canale destro: {rms_right:.6f}")

In [ ]:
AUDIO_DIR = Path("file_wav")

risultati_canali = []

for wav_path in sorted(AUDIO_DIR.glob("*.wav")):
    audio_corrente, sr_corrente = sf.read(wav_path)

    # Gestione del file mono
    if audio_corrente.ndim != 2 or audio_corrente.shape[1] != 2:
        risultati_canali.append({
            "file": wav_path.name,
            "sample_rate": sr_corrente,
            "canali": 1,
            "correlazione": None,
            "rms_sinistro": None,
            "rms_destro": None,
            "rapporto_rms_sx_dx": None,
            "canale_piu_energetico": "mono"
        })
        continue

    canale_sinistro = audio_corrente[:, 0]
    canale_destro = audio_corrente[:, 1]

    correlazione = np.corrcoef(
        canale_sinistro,
        canale_destro
    )[0, 1]

    rms_sinistro = np.sqrt(np.mean(canale_sinistro ** 2))
    rms_destro = np.sqrt(np.mean(canale_destro ** 2))

    risultati_canali.append({
        "file": wav_path.name,
        "sample_rate": sr_corrente,
        "canali": 2,
        "correlazione": correlazione,
        "rms_sinistro": rms_sinistro,
        "rms_destro": rms_destro,
        "rapporto_rms_sx_dx": (
            rms_sinistro / rms_destro
            if rms_destro > 0
            else None
        ),
        "canale_piu_energetico": (
            "sinistro"
            if rms_sinistro > rms_destro
            else "destro"
        )
    })

df_canali = pd.DataFrame(risultati_canali)

display(df_canali)

In [ ]:
df_canali["canale_piu_energetico"].value_counts()

In [ ]:
df_canali["correlazione"].describe()

In [ ]:
df_canali["rapporto_rms_sx_dx"].describe()

In [ ]:
df_canali.sort_values("correlazione").head(10)

In [ ]:
Path("risultati").mkdir(exist_ok=True)

df_canali.to_csv(
    "risultati/analisi_canali.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Risultati salvati correttamente")

In [ ]:
stereo = df_canali[df_canali["canali"] == 2].copy()

file_correlazione_minima = stereo.nsmallest(
    1, "correlazione"
)

file_correlazione_massima = stereo.nlargest(
    1, "correlazione"
)

file_sinistro_dominante = stereo.nlargest(
    1, "rapporto_rms_sx_dx"
)

file_destro_dominante = stereo.nsmallest(
    1, "rapporto_rms_sx_dx"
)

file_correlazione_mediana = stereo.iloc[
    (stereo["correlazione"] - stereo["correlazione"].median())
    .abs()
    .argsort()[:1]
]

campione = pd.concat([
    file_correlazione_minima,
    file_correlazione_massima,
    file_sinistro_dominante,
    file_destro_dominante,
    file_correlazione_mediana
]).drop_duplicates("file")

display(campione)

In [ ]:
from IPython.display import Audio, display
import soundfile as sf
from pathlib import Path

AUDIO_DIR = Path("file_wav")

def ascolta_segmenti_canali(nome_file, durata_segmento=20):
    percorso = AUDIO_DIR / nome_file
    audio, sr = sf.read(percorso)

    if audio.ndim != 2 or audio.shape[1] != 2:
        print(f"{nome_file}: file mono")
        display(Audio(audio, rate=sr))
        return

    durata_totale = len(audio) / sr

    punti = {
        "inizio": 0,
        "centro": max(0, durata_totale / 2 - durata_segmento / 2),
        "fine": max(0, durata_totale - durata_segmento)
    }

    print(f"\nFile: {nome_file}")
    print(f"Durata totale: {durata_totale:.2f} secondi")

    for posizione, inizio_sec in punti.items():
        inizio = int(inizio_sec * sr)
        fine = min(
            inizio + int(durata_segmento * sr),
            len(audio)
        )

        sinistro = audio[inizio:fine, 0]
        destro = audio[inizio:fine, 1]

        print(
            f"\n{posizione.upper()} "
            f"({inizio_sec:.1f}-{fine / sr:.1f} s)"
        )

        print("Canale sinistro")
        display(Audio(sinistro, rate=sr))

        print("Canale destro")
        display(Audio(destro, rate=sr))

In [ ]:
ascolta_segmenti_canali("ID89.StudioRuggi.wav")

In [ ]:
ascolta_segmenti_canali("ID20.StudioRuggi.wav")

In [ ]:
ascolta_segmenti_canali("ID130.StudioRuggi.wav")

In [ ]:
ascolta_segmenti_canali("ID1.StudioRuggi-006.wav")

In [ ]:
ascolta_segmenti_canali("ID26.StudioRuggi.wav")

In [ ]:
ascolta_segmenti_canali("ID16.StudioRuggi.wav")

In [ ]:
ascolta_segmenti_canali("ID16.StudioRuggi(1).wav")

In [ ]:
ascolta_segmenti_canali("ID36.StudioRuggi.wav")

In [ ]:
ascolta_segmenti_canali("ID36.StudioRuggi(2).wav")

### Esito dell’ascolto delle registrazioni con ID duplicato

In [ ]:
audit_duplicati = pd.DataFrame([
    {
        "id": 16,
        "file_1": "ID16.StudioRuggi.wav",
        "file_2": "ID16.StudioRuggi(1).wav",
        "voce_non_paziente": "dottoressa",
        "esito_ascolto": "contenuti non identici",
        "similarita_massima": 0.2821,
        "criticita": (
            "nessuna anomalia specifica rilevata "
            "nei segmenti ascoltati"
        ),
        "decisione": (
            "conservare separati: non emerge evidenza "
            "che il file breve sia contenuto nel file lungo"
        )
    },
    {
        "id": 36,
        "file_1": "ID36.StudioRuggi.wav",
        "file_2": "ID36.StudioRuggi(2).wav",
        "voce_non_paziente": "voce preregistrata",
        "esito_ascolto": "contenuti non identici",
        "similarita_massima": 0.3103,
        "criticita": (
            "volume molto basso all'inizio del canale sinistro; "
            "rumore elevato, soprattutto alla fine del canale "
            "sinistro di ID36.StudioRuggi(2).wav"
        ),
        "decisione": (
            "conservare separati: non emerge evidenza "
            "che il file breve sia contenuto nel file lungo"
        )
    }
])

display(audit_duplicati)

In [ ]:
audit_duplicati.to_csv(
    "risultati/audit_id_duplicati.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Analisi degli ID duplicati salvata correttamente")

In [ ]:
from scipy.signal import correlate, resample_poly
from math import gcd

def carica_canale_piu_energetico(nome_file, target_sr=8000):
    audio, sr = sf.read(
        AUDIO_DIR / nome_file,
        always_2d=True
    )

    rms_canali = np.sqrt(np.mean(audio ** 2, axis=0))
    indice_canale = int(np.argmax(rms_canali))

    segnale = audio[:, indice_canale]

    if sr != target_sr:
        divisore = gcd(sr, target_sr)

        segnale = resample_poly(
            segnale,
            target_sr // divisore,
            sr // divisore
        )

    return segnale, target_sr


def calcola_inviluppo_rms(
    segnale,
    sr,
    frame_ms=100,
    hop_ms=50
):
    frame = int(sr * frame_ms / 1000)
    hop = int(sr * hop_ms / 1000)

    valori_rms = []

    for inizio in range(0, len(segnale) - frame + 1, hop):
        finestra = segnale[inizio:inizio + frame]
        rms = np.sqrt(np.mean(finestra ** 2))
        valori_rms.append(rms)

    return np.asarray(valori_rms), hop_ms / 1000


def verifica_contenimento(file_breve, file_lungo):
    breve, sr = carica_canale_piu_energetico(file_breve)
    lungo, _ = carica_canale_piu_energetico(file_lungo)

    env_breve, passo_sec = calcola_inviluppo_rms(breve, sr)
    env_lungo, _ = calcola_inviluppo_rms(lungo, sr)

    if len(env_breve) > len(env_lungo):
        raise ValueError(
            "Il primo file deve essere quello più breve."
        )

    breve_norm = (
        env_breve - np.mean(env_breve)
    ) / (np.std(env_breve) + 1e-12)

    lungo_norm = (
        env_lungo - np.mean(env_lungo)
    ) / (np.std(env_lungo) + 1e-12)

    correlazioni = correlate(
        lungo_norm,
        breve_norm,
        mode="valid",
        method="fft"
    )

    posizione = int(np.argmax(correlazioni))

    tratto_lungo = env_lungo[
        posizione:posizione + len(env_breve)
    ]

    similarita = np.corrcoef(
        env_breve,
        tratto_lungo
    )[0, 1]

    inizio_secondi = posizione * passo_sec
    fine_secondi = (
        inizio_secondi + len(breve) / sr
    )

    return {
        "file_breve": file_breve,
        "file_lungo": file_lungo,
        "similarita_massima": round(float(similarita), 4),
        "possibile_inizio_nel_lungo": round(inizio_secondi, 2),
        "possibile_fine_nel_lungo": round(fine_secondi, 2)
    }

In [ ]:
risultato_id16 = verifica_contenimento(
    "ID16.StudioRuggi.wav",
    "ID16.StudioRuggi(1).wav"
)

risultato_id16

In [ ]:
risultato_id36 = verifica_contenimento(
    "ID36.StudioRuggi.wav",
    "ID36.StudioRuggi(2).wav"
)

risultato_id36